ESSENTIAL IMPORTS

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

import joblib

LOADING THE DATASET

In [2]:
df = pd.read_csv("../data/loan_approval_1000.csv")
display(df.head())
print(df.shape)

,Age,Income,Loan_Amount,Credit_Score,Employment_Type,Loan_Approved
0,59,77761.0,417505,337,Salaried,0
1,49,141178.0,328226,487,Salaried,0
2,35,56368.0,312805,356,Self-Employed,0
3,28,26783.0,218484,319,Salaried,0
4,41,131530.0,315064,572,Self-Employed,0


(1000, 6)


In [3]:
print("Info")
df.info()
print("Missing values")
print(df.isnull().sum())


Info
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Age              1000 non-null   int64  
 1   Income           950 non-null    float64
 2   Loan_Amount      1000 non-null   int64  
 3   Credit_Score     1000 non-null   int64  
 4   Employment_Type  950 non-null    object 
 5   Loan_Approved    1000 non-null   int64  
dtypes: float64(1), int64(4), object(1)
memory usage: 47.0+ KB
Missing values
Age                 0
Income             50
Loan_Amount         0
Credit_Score        0
Employment_Type    50
Loan_Approved       0
dtype: int64


In [4]:
print("Statistical Summary")
print(df.describe(include="all"))

Statistical Summary
                Age         Income    Loan_Amount  Credit_Score  \
count   1000.000000     950.000000    1000.000000   1000.000000   
unique          NaN            NaN            NaN           NaN   
top             NaN            NaN            NaN           NaN   
freq            NaN            NaN            NaN           NaN   
mean      40.445000   84881.753684  270051.753000    610.381000   
std       11.499183   37717.819324  128863.845332    174.696974   
min       21.000000   20060.000000   51969.000000    300.000000   
25%       30.000000   51909.000000  160877.250000    460.000000   
50%       41.000000   83551.500000  266606.500000    612.000000   
75%       50.000000  116955.750000  383224.500000    764.000000   
max       59.000000  149972.000000  499370.000000    898.000000   

       Employment_Type  Loan_Approved  
count              950    1000.000000  
unique               3            NaN  
top           Salaried            NaN  
freq           

In [5]:
print("Categorical Values")
print(df["Employment_Type"].value_counts())

Categorical Values
Employment_Type
Salaried         471
Self-Employed    287
Unemployed       192
Name: count, dtype: int64


In [6]:
print("Target Distribution")
print(df["Loan_Approved"].value_counts())

Target Distribution
Loan_Approved
0    685
1    315
Name: count, dtype: int64


In [7]:
print("Duplicate Records:")
print(df.duplicated().sum())

Duplicate Records:
0


DATA PREPROCESSING

In [8]:
x=df.drop("Loan_Approved",axis=1)
y=df["Loan_Approved"]
numeric_features = [
    "Age",
    "Income",
    "Loan_Amount",
    "Credit_Score"
]
categorical_features = [
    "Employment_Type"
]
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])


TRAIN/TEST SPLIT

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))

Training records: 800
Testing records: 200


MODEL TRAINING

In [13]:
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42))
])
model.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


PREDICTION PROBABILITIES

In [15]:
y_pred=model.predict(X_test)
y_prob = model.predict_proba(X_test)
print("Accuracy: ", accuracy_score(y_test, y_pred))
print("Precision: ", precision_score(y_test, y_pred))
print("Recall: ", recall_score(y_test, y_pred))
print("F1 Score: ", f1_score(y_test, y_pred))
cm = confusion_matrix(y_test, y_pred)
print(cm)

Accuracy:  0.795
Precision:  0.7037037037037037
Recall:  0.6031746031746031
F1 Score:  0.6495726495726496
[[121  16]
 [ 25  38]]


SAVING THE MODEL

In [16]:
joblib.dump(model, "../models/loan_approval_model.pkl")
print("Model saved successfully.")

Model saved successfully.


In [17]:
import os
print(
    os.path.exists(
        "../models/loan_approval_model.pkl"
    )
)

True


TESTING BY LOADING THE MODEL

In [19]:
loaded_model=joblib.load("../models/loan_approval_model.pkl")
loaded_pred=loaded_model.predict(X_test)
print("Same predictions: ",(loaded_pred==y_pred).all())

Same predictions:  True
